# Libraries

In [0]:
%pip install openpyxl
%restart_python

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

# Data

## Reading

In [0]:
directory = '/Volumes/data_exploration_hub/marketing_hub/model_data/v1/raw_data/' # Change to the directory of the data

# Website visits (Cineplex, Rec Room, Playdium)
cineplex_web = pd.read_csv(directory + 'Cineplex dot com - Day.csv', skiprows=11)
recroom_web = pd.read_csv(directory + 'TheRecRoom.com - Day.csv', skiprows=11)
playdium_web = pd.read_csv(directory + 'Playdium Production - Day.csv', skiprows=11)

# Cineclub signups and exits
names = ['date', 'monthly_signups', 'monthly_exits', 'annual_signups', 'annual_exits', 'gift_signups', 'gift_exits']
cineclub = pd.read_csv(directory + 'cineclub_signups.csv', names=names)

# Attendance and revenue
attendance_and_revenue = pd.read_csv(directory + 'attendance_and_revenue.csv')

# SEM spend
sem = pd.read_excel(directory + 'Cineplex SEM 2023- 2025 raw data .xlsx', skiprows=2)
sem_new = pd.read_excel(directory + 'SEM EXH & LBE Mar 2023 to Jun 2025 weekly report_v2.xlsx', skiprows=2)

# Meta spend (EXH and LBE)
meta_exh = pd.read_excel(directory + 'META EXH Jan 2023 to Apr 2025 weekly report.xlsx')
meta_exh_new = pd.read_excel(directory + 'META EXH Mar 2023 to June 2025 weekly report_v2.xlsx')
meta_lbe = pd.read_excel(directory + 'META LBE Jan 2023 to Apr 2025 weekly report.xlsx')

# Datorama channels report
datorama = pd.read_excel(directory + 'Datorama Report All Channels Jan 2023 to Apr 2025 Daily Report.xlsx', skiprows=4, skipfooter=1)

# Email campaigns
email_2023_h1 = pd.read_excel(directory + '2023 YTD Email Performance - June 28.xlsx', sheet_name='All Email Performance', skipfooter=10)
email_2023_h2 = pd.read_excel(directory + '(MASTER) Email Performance Report - H2 2023.xlsx', skipfooter=1)
email_2024 = pd.read_excel(directory + '(MASTER) Email Performance Report_2024.xlsx', sheet_name='All AJO Emails', skipfooter=1)
email_2025 = pd.read_excel(directory + '(MASTER) Email Performance Report_2025 - May 28.xlsx', sheet_name='YTD - All AJO Emails', skipfooter=1)

# PR spend
pr = pd.read_csv(directory + 'pr.csv')

# Film releases
releases = pd.read_csv(directory + 'film_releases.csv')

# Mass offer calendar
mass_offers = pd.read_csv(directory + 'mass_offers.csv')

## Cleaning

In [0]:
# Cleaning column names
dfs = [cineplex_web, recroom_web, playdium_web,
       attendance_and_revenue, cineclub, sem, sem_new,
       meta_exh, meta_exh_new, meta_lbe, datorama, 
       email_2023_h1, email_2023_h2, email_2024, 
       email_2025, releases]

for df in dfs:
    df.columns = df.columns.str.lower().str.replace(' ', '_')

In [0]:
# Defining new column labels
new_cols = ['date', 'visits', 'unique_visitors', 'orders']

# Renaming columns
cineplex_web.columns = new_cols
recroom_web.columns = new_cols
playdium_web.columns = new_cols

# Combining website data
website = pd.concat([cineplex_web[['date', 'visits']], recroom_web['visits'], playdium_web['visits']], axis=1)
website.columns = ['date', 'cineplex_visits', 'recroom_visits', 'playdium_visits']

# Converting data types
website['date'] = pd.to_datetime(website['date'])
website['week'] = website['date'].dt.to_period('W-THU').dt.start_time
website = website.drop('date', axis=1)

website.sample()

In [0]:
# Converting data types
cineclub['date'] = pd.to_datetime(cineclub['date'], errors='coerce')
cineclub['week'] = cineclub['date'].dt.to_period('W-THU').dt.start_time
cineclub = cineclub.drop('date', axis=1)

cineclub.sample()

In [0]:
# Defining new column labels
new_cols = ['date', 'attendance', 'total_revenue', 'box_office_revenue', 'concessions_revenue', 'lbe_revenue']

# Renaming columns
attendance_and_revenue.columns = new_cols

# Converting data types
attendance_and_revenue['attendance'] = attendance_and_revenue['attendance'].astype('int64')
attendance_and_revenue['date'] = pd.to_datetime(attendance_and_revenue['date'])
attendance_and_revenue['week'] = attendance_and_revenue['date'].dt.to_period('W-THU').dt.start_time
attendance_and_revenue = attendance_and_revenue.drop('date', axis=1)

attendance_and_revenue.sample()

In [0]:
# Renaming columns
sem['lob'] = np.where(sem['lob'].str.contains('Prepaid'), 'Prepaid', sem['lob'])
sem['lob'] = np.where(sem['lob'].str.contains('IMAX|New Screens'), 'IMAX', sem['lob'])

# Converting data types
if 'day' in sem.columns: sem['day'] = pd.to_datetime(sem['day'])
sem['week'] = pd.to_datetime(sem['week'])

# Resetting week start date
if 'day' in sem.columns: sem['week'] = sem['day'].dt.to_period('W-THU').dt.start_time
if 'day' in sem.columns: sem = sem.drop('day', axis=1)

sem.sample()

In [0]:
# Combining Meta spend data
meta_exh['frequency'] = meta_exh['impressions'] / meta_exh['reach']
meta_exh['type'] = 'EXH'
meta_lbe['type'] = 'LBE'
meta = pd.concat([meta_exh, meta_lbe]).drop(columns=['unnamed:_10', 'shop_clicks'])

# Converting data types
meta['reporting_starts'] = pd.to_datetime(meta['reporting_starts'])
meta['week'] = meta['reporting_starts'].dt.to_period('W-THU').dt.start_time
meta = meta.drop('reporting_starts', axis=1)

# Keeping relevant observations
meta = meta.dropna(subset='amount_spent_(cad)')

meta.sample()

In [0]:
# Converting data types
datorama['day'] = pd.to_datetime(datorama['day'])
datorama['week'] = datorama['day'].dt.to_period('W-THU').dt.start_time
datorama = datorama.drop('day', axis=1)

# Splitting data
programmatic = datorama[datorama['channel']!='Social']
social = datorama[(datorama['channel']=='Social')*(~datorama['publisher'].str.contains('Meta'))]

datorama.sample()

In [0]:
# Setting column names to select
cols = {'send_date': 'send_date', 'email_name': 'journey_name', 
        'audience': 'version_language', 'sends': 'sent', 
        'unique_opens': 'unique_opens', 'open_rate': 'unique_open_rate', 
        'unique_clicks': 'unique_clicks', 'cto': 'cto', 'ctr': 'ctr'}

# Renaming columns
email_2023_h1.rename(columns=cols, inplace=True)

# Combining email data
email = pd.concat([email_2023_h1[cols.values()], email_2023_h2, email_2024, email_2025])

# Converting data types
email['send_date'] = pd.to_datetime(email['send_date'])
email['week'] = email['send_date'].dt.to_period('W-THU').dt.start_time
email = email.drop('send_date', axis=1)

email.sample()

In [0]:
# Converting data types
pr['week'] = pd.to_datetime(pr['week'])
pr['week'] = pr['week'].dt.to_period('W-THU').dt.start_time

pr.sample()

In [0]:
# Converting data types
releases['release_date'] = pd.to_datetime(releases['release_date'])
releases['week'] = releases['release_date'].dt.to_period('W-THU').dt.start_time
releases = releases.drop('release_date', axis=1)

releases.sample()

In [0]:
# Converting data types
mass_offers['week'] = pd.to_datetime(mass_offers['week'])
mass_offers['week'] = mass_offers['week'].dt.to_period('W-THU').dt.start_time

mass_offers.sample()

In [0]:
# # Exporting clean data
# directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/clean_data/'

# website.to_csv(directory + 'website.csv', index=False)
# attendance_and_revenue.to_csv(directory + 'attendance_and_revenue.csv', index=False)
# cineclub.to_csv(directory + 'cineclub.csv', index=False)
# sem.to_csv(directory + 'sem.csv', index=False)
# meta.to_csv(directory + 'meta.csv', index=False)
# programmatic.to_csv(directory + 'programmatic.csv', index=False)
# social.to_csv(directory + 'social.csv', index=False)
# email.to_csv(directory + 'email.csv', index=False)
# releases.to_csv(directory + 'releases.csv', index=False)
# pr.to_csv(directory + 'pr.csv', index=False)
# mass_offers.to_csv(directory + 'mass_offers.csv', index=False)

# Summary Statistics

In [0]:
# Website visits
website.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Cineclub signups and exits
cineclub.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Attendance and revenue
attendance_and_revenue.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# SEM spend
sem.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Meta spend
meta.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Datorama programmatic
programmatic.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Datorama social
social.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Email campaigns
email.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# PR spend
pr.select_dtypes(['int64', 'float64']).describe().round(2)

In [0]:
# Mass offer calendar
mass_offers.select_dtypes(['int64', 'float64']).describe().round(2)

# Feature Engineering

## Preprocessing

In [0]:
# Defining objective table to export (last cell in notebook)
objective = 'Conversion'

In [0]:
# Adding objectives to channels
cols = ['Awareness', 'Consideration', 'Conversion']

def cases(column: pd.Series):
    cases = [column.str.contains('Awareness|_AW|reach', na=False), 
            column.str.contains('Consideration|_CONS|_CN|link_click|landing_page_view', na=False),
            column.str.contains('Conversion|_CVN|_CVS|001_CP2K790|offsite_conversion|subscribe_website', na=False)]
    return cases

sem['objective'] = np.select(cases(sem['campaign']), cols, default='')
meta['objective'] = np.select(cases(meta['result_indicator']), cols, default='')
programmatic = programmatic[programmatic['objective'].isin(cols)]
social = social[social['objective'].isin(cols)]
email['objective'] = 'Conversion'

In [0]:
# Mapping objectives to lines of business
objective_lobs = {
    'Awareness': [[], 
                  ['BRA', 'TFB'], 
                  ['Brand', 'Cineclub'], 
                  []],
    'Consideration': [['Exhibition ', 'G&E'], 
                      ['BRA', 'GRP'], 
                      ['Brand', 'Enhanced Experiences', 'Film Marketing', 'Prepaid'], 
                      ['Groups & Events (G&E)']],
    'Conversion': [['Cineclub', 'Exhibition ', 'IMAX'], 
                   ['A&C', 'BRA', 'FLM'], 
                   ['Brand', 'Enhanced Experiences', 'Performance'], 
                   ['Film Marketing', 'Performance']]
}

# Mapping lines of business to cleaner names
clean_lobs = {
    'Awareness': [[], 
                  ['brand', 'fnb'], 
                  ['brand', 'cineclub'], 
                  []],
    'Consideration': [['exhibition', 'gne'], 
                      ['brand', 'gne'], 
                      ['brand', 'enhanced', 'film', 'prepaid'], 
                      ['gne']],
    'Conversion': [['cineclub', 'exhibition', 'enhanced'], 
                   ['event', 'brand', 'film'], 
                   ['brand', 'enhanced', 'performance'], 
                   ['film', 'performance']]
}

In [0]:
# Website, cineclub, attendance and revenue
temp1 = pd.concat([website.drop('week', axis=1), cineclub, attendance_and_revenue.drop('week', axis=1)], axis=1)
temp1['website_visits'] = temp1[['cineplex_visits', 'recroom_visits', 'playdium_visits']].sum(axis=1)
temp1['cineclub_signups'] = temp1[['monthly_signups', 'annual_signups', 'gift_signups']].sum(axis=1)
cols = ['week', 'website_visits', 'cineclub_signups', 'attendance']
temp1 = temp1[cols].groupby('week').sum()

# SEM spend
cols = objective_lobs[objective][0]
temp2 = sem[sem['objective']==objective].copy()
temp2['lob'] = np.where(temp2['lob'].str.contains('Prepaid'), 'Prepaid', temp2['lob'])
temp2 = temp2[temp2['lob'].isin(cols)][['week', 'lob', 'cost']]
temp2 = temp2.groupby(['week', 'lob']).agg({'cost': 'sum'}).unstack().fillna(0)
temp2 = temp2.droplevel(0, axis=1)
temp2.index.name = None
temp2.columns = ['sem_' + lob for lob in clean_lobs[objective][0]]

# Meta spend
temp3 = meta[meta['objective']==objective].copy()
temp3['campaign_short'] = temp3['campaign_name'].str.extract(r'(?:CEC|CEA)_(.*?)_')
cols = objective_lobs[objective][1]
temp3 = temp3[temp3['campaign_short'].isin(cols)][['week', 'campaign_short', 'amount_spent_(cad)']]
temp3 = temp3.groupby(['week', 'campaign_short']).agg({'amount_spent_(cad)': 'sum'}).unstack().fillna(0)
temp3 = temp3.droplevel(0, axis=1)
temp3.index.name = None
temp3.columns = ['meta_' + lob for lob in clean_lobs[objective][1]]

# Datorama programmatic
cols = objective_lobs[objective][2]
temp4 = programmatic[programmatic['objective']==objective].copy()
temp4['line_of_business'] = np.where(temp4['line_of_business']=='B2C', 'Prepaid', temp4['line_of_business'])
temp4 = temp4[temp4['line_of_business'].isin(cols)][['week', 'line_of_business', 'investment']]
temp4 = temp4.groupby(['week', 'line_of_business']).agg({'investment': 'sum'}).unstack().fillna(0)
temp4 = temp4.droplevel(0, axis=1)
temp4.index.name = None
temp4.columns = ['programmatic_' + lob for lob in clean_lobs[objective][2]]

# Datorama social
cols = objective_lobs[objective][3]
temp5 = social[social['objective']==objective].copy()
temp5 = temp5[temp5['line_of_business'].isin(cols)][['week', 'line_of_business', 'investment']]
temp5 = temp5.groupby(['week', 'line_of_business']).agg({'investment': 'sum'}).unstack().fillna(0)
temp5 = temp5.droplevel(0, axis=1)
temp5.index.name = None
temp5.columns = ['social_' + lob for lob in clean_lobs[objective][3]]

# # Email campaigns
# temp6 = email[email['week']<pd.to_datetime('2025-04-30')].copy()
# temp6 = temp6[temp6['objective']=='Conversion']
# conditions = [
#     (temp6['journey_name'].str.contains('CineClub|Cine Club')),
#     (temp6['journey_name'].str.contains('MKTB|MKBT|MKTX|Scene+')),
#     (temp6['journey_name'].str.contains('MKTT|EVCN|MKTI'))
# ]
# choices = ['CineClub', 'Brand', 'Film']
# temp6['lob'] = np.where(temp6['lob'].isna(), np.select(conditions, choices, default=temp6['lob']), temp6['lob'])
# cols = ['Film', 'Brand', 'CineClub']
# temp6 = temp6[temp6['lob'].isin(cols)][['week', 'lob', 'sent']]
# temp6 = temp6.groupby(['week', 'lob']).agg({'sent': 'sum'}).unstack().fillna(0)
# temp6 = temp6.droplevel(0, axis=1)
# temp6.index.name = None
# temp6.columns = 'email_' + temp6.columns.str.lower()

# PR spend
temp7 = pr.set_index('week').copy()
temp7['pr_spend'] = temp7.sum(axis=1)
temp7 = temp7[['pr_spend']]

# Film releases
temp8 = releases.copy()
temp8 = temp8[['week', 'title_id']].groupby('week').count()
temp8.index.name = None
temp8.columns = ['releases']

# Mass offer calendar
temp9 = mass_offers.set_index('week').copy()

In [0]:
# Combining all temporary dataframes
df = pd.concat([temp1, temp2, temp3, temp4, temp5, temp7, temp8, temp9], axis=1).fillna(0)
df = df[(df.index >= '2023-03-31') & (df.index <= '2025-03-28')]

In [0]:
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(df.corr(), annot=True, mask=np.triu(np.ones_like(df.corr(), dtype=bool)))
plt.show()

## Feature Creation

In [0]:
# Defining holidays
holidays_by_name = {
    "New Year's Day": ['2023-01-01', '2024-01-01', '2025-01-01'],
    'Family Day': ['2023-02-20', '2024-02-19', '2025-02-17'],
    'Good Friday': ['2023-04-07', '2024-03-29', '2025-04-18'],
    'Easter Monday': ['2023-04-10', '2024-04-01', '2025-04-21'],
    'Victoria Day': ['2023-05-22', '2024-05-20', '2025-05-19'],
    'Canada Day': ['2023-07-01', '2024-07-01', '2025-07-01'],
    'Labour Day': ['2023-09-04', '2024-09-02', '2025-09-01'],
    'Thanksgiving': ['2023-10-09', '2024-10-14', '2025-10-13'],
    'Christmas Day': ['2023-12-25', '2024-12-25', '2025-12-25']
}

holidays = [pd.to_datetime(date) for date_list in holidays_by_name.values() for date in date_list]
holidays = pd.Series(holidays).dt.to_period('W-THU').dt.start_time

# Creating new columns
df['summer'] = np.where(df.index.month.isin([7, 8]), 1, 0)
df['holiday'] = np.where(df.index.isin(holidays), 1, 0)

In [0]:
# Reordering columns
kpi_cols = list(df.columns[:3])
spend_cols = list(df.columns[3:-6])
season_cols = list(df.columns[-6:])

df = df[season_cols + spend_cols + kpi_cols]
df.index.name = 'week'

In [0]:
# Exporting model data
directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/model_data/' # Change to the directory of the data

df.to_csv(directory + f'exh_{objective.lower()}_mmm.csv') # Change 'objective' to export different model data